# **LLM Pipeline to Process Extracted Dataset (Phase 1)**

In [ ]:
!pip install pandas

### **Start custom worker on MultiCare dataset**

In [1]:
from DatasetWorker import mount_worker
from DatasetWorkerHelpers import get_secret

DATASET_NAME = "extracted_dataset_1792212375"
worker = mount_worker(DATASET_NAME)

Worker mounted on: extracted_dataset_1792212375 (renamed 0 cases.csv, 13 already normalized)


/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Clean up junk files (safe to run everytime)**

In [2]:
worker.clean_dataset()

### **Eliminate cases with ambiguous demographic**

In [ ]:
worker.delete_unknown_demographics()

### **Generate .txt queries for every cases.csv found**

In [ ]:
worker.generate_diagnosis_questions()

### **Attach scores to cases (requiring prompt 3 and result.json)**

In [ ]:
worker.attach_scores()

### **Show dataset quality metrics**

In [ ]:
worker.display_metrics()

# **Final Processing (Split into train and test)**

### **Build big dataset (Generate `normalized_diseases.csv` and `evaluated_cases.csv`)**

In [ ]:
worker.build_master_dataset()

### **Split into train and test. Mask test set (use prompt 4)**

In [7]:
train_df, test_df = worker.assign_splits()

masked_df = worker.get_masked_test_cases()

Another path: Structured cases

In [3]:
train_df, test_df, masked_df = worker.read_soap_notes()


,case_id,pair_id,split,target_disease,gender,age,raw_case_text,soap_note
0,PMC9892713_01,PMC9892713_01::Brucellosis,test_masked,Brucellosis,Male,79,A 79-year-old male was admitted to a hospital ...,{'subjective': {'chief_complaint': 'Four-patie...
1,PMC9767972_01,PMC9767972_01::Brucellosis,test_masked,Brucellosis,Female,51,"On 24 May 2021, a 51-year-old woman was admitt...",{'subjective': {'chief_complaint': 'A 51-year-...
2,PMC6639843_01,PMC6639843_01::Brucellosis,test_masked,Brucellosis,Male,19,A 19-year-old male patient was referred to the...,{'subjective': {'chief_complaint': 'A 19-year-...
3,PMC10921472_01,PMC10921472_01::Brucellosis,test_masked,Brucellosis,Male,55,A 55-year-old male was admitted to our hospita...,{'subjective': {'chief_complaint': 'A 55-year-...
4,PMC11746083_04,PMC11746083_04::Brucellosis,test_masked,Brucellosis,Male,26,A 26-year-old male presented with headache alo...,{'subjective': {'chief_complaint': 'A 26-year-...
...,...,...,...,...,...,...,...,...
68,PMC12858201_01,PMC12858201_01::Pulmonary Tuberculosis,test_masked,Pulmonary Tuberculosis,Female,17,A 17-year-old female had been dealing with rep...,{'subjective': {'chief_complaint': 'A 17-year-...
69,PMC9112304_01,PMC9112304_01::Q Fever,test_masked,Q Fever,Male,53,A 53-year-old male patient was referred to the...,{'subjective': {'chief_complaint': 'A 53-year-...
70,PMC5803979_03,PMC5803979_03::Q Fever,test_masked,Q Fever,Female,3,"AC is a 3-year-old girl, a case of Tetralogy o...",{'subjective': {'chief_complaint': 'A 3-year-o...
71,PMC11165120_01,PMC11165120_01::Q Fever,test_masked,Q Fever,Male,73,"A 73-year-old man, with a medical history of n...",{'subjective': {'chief_complaint': 'A 73-year-...


### **Upload to HuggingFace**

In [4]:
from huggingface_hub import login, whoami

# Retrieve token and set env var in one line
hf_token = get_secret("HF_TOKEN")

login(token=hf_token)
print(whoami())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'type': 'user', 'id': '6aad11231339b5a189922a50', 'name': 'choisongjun', 'fullname': 'HT', 'isPro': False, 'avatarUrl': '/avatars/ebdd5c23e3145dbd98658c6e73fa3616.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HuggingFace-CreateDataset', 'role': 'fineGrained', 'createdAt': '2026-10-08T04:12:35.840Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6aad11231339b5a189922a50', 'type': 'user', 'name': 'choisongjun'}, 'permissions': ['repo.contribute.write', 'repo.content.read', 'repo.access.read', 'repo.write', 'discussion.write']}]}}}}


In [5]:
from DatasetWorkerHelpers import build_hf_dataset

dataset_dict = build_hf_dataset(train_df=train_df, test_df=test_df, masked_df=masked_df)
REPO_ID = "choisongjun/Tropical_Diseases_Repo_v3.1"

In [6]:
dataset_dict.push_to_hub(REPO_ID, token=hf_token)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 71.89ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  37%|███▋      |  527kB / 1.42MB, 48.4kB/s  
Processing Files (0 / 1):  74%|███████▍  | 1.05MB / 1.42MB, 70.7kB/s  
Processing Files (0 / 1): 100%|█████████▉| 1.42MB / 1.42MB, 91.1kB/s  
Processing Files (1 / 1): 100%|██████████| 1.42MB / 1.42MB, 82.7kB/s  
New Data Upload: 100%|██████████| 1.42MB / 1.42MB, 82.7kB/s  

Validating:           |   0%            

Validating:           |   0%            

Validating:           |   0%            

Validating:           |   0%            

Processing Files (1 / 1): 100%|██████████| 1.42MB / 1.42MB, 80.8kB/s  
New Data Upload: 100%|██████████| 1.42MB / 1.42MB, 80.8kB/s  
Validating: ██████████| 100%            
Uploading the dataset shards: 100%|██████████| 1/1


Pushed to https://huggingface.co/datasets/choisongjun/Tropical_Diseases_Repo_v3.1
